<a href="https://colab.research.google.com/github/replysantosh-lang/ECARAgenticAI/blob/main/17_Agentic_Web_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Agentic Search

In [1]:
# libraries
!pip install tavily-python
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [2]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


"Nvidia's new Blackwell GPU architecture features 208 billion transistors and is manufactured using a custom TSMC 4NP process, offering significant improvements in AI compute performance and efficiency. The architecture includes the B100 and B200 datacenter accelerators, promising up to 50x higher performance per megawatt."

## Regular search

In [3]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [4]:
!pip install ddgs

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results


for i in search(query):
    print(i)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 66.9 MB/s eta 0:00:00
https://weather.com/us/california/city/san-francisco/today
https://weather.com/us/california/city/san-francisco/tenday
https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8
https://weather.com/weather/today/l/San+Francisco+CA+USCA0987:1:US
https://www.weather-us.com/en/california-usa/san-francisco
https://weather.com/weather/today/l/South+San+Francisco+CA?canonicalCityId=d9909813b4b077bd115f4e969590f69a


In [5]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."

    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [6]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

Website: https://weather.com/us/california/city/san-francisco/today


<body class="inter_3e69cff8-module__PPczxG__className font-sans"><script>(self.__next_s=self.__next_s||[]).push([0,{"children":"(function(){\n        try{\n          var cookies=document.cookie.split('; ');\n          var sessionCookie='';\n          for(var i=0;i\u003ccookies.length;i++){\n            if(cookies[i].indexOf('wxu-metrics-session=')===0){\n              sessionCookie=cookies[i].slice(20);break;\n            }\n          }\n          var now=Date.now();\n          var la=parseInt(localStorage.getItem('metric-session-last-active-time')||'0',10);\n          var expired=la\u003e0\u0026\u0026Math.abs(now-la)\u003e1800000;\n          if(!sessionCookie||expired){\n            var id=crypto\u0026\u0026crypto.randomUUID ? crypto.randomUUID() : (function(){\n              return 'xxxxxxxx-xxxx-4xxx-yxxx-xxxxxxxxxxxx'.replace(/[xy]/g,function(c){\n                var r=Math.random()*16|0;return(c==='x'?r:(r\u0026

In [7]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)

print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/us/california/city/san-francisco/today


Forecasts Radar Explore More Sign in Get Premium Upgrade Feels Like -- H High -- L Low -- Chance of Rain 0% Today's Outlook Daily Forecast Trending Now Why so much severe weather pounded the Northeast and Midwest over the Fourth of July weekend Going to a lake or river? Check for these hidden dangers first 0:43 2026 Atlantic hurricane season forecast to be least active in over a decade amid El Niño, CSU says You won't believe what this claw fished out of a flood 0:24 Why so much severe weather pounded the Northeast and Midwest over the Fourth of July weekend Going to a lake or river? Check for these hidden dangers first 0:43 2026 Atlantic hurricane season forecast to be least active in over a decade amid El Niño, CSU says You won't believe what this claw fished out of a flood 0:24 Editor's Pick Hundreds of homes lost as massive wildfires rage in Colorado, Utah Dry, gusty winds are fueling dangerous fire conditions ac

## Agentic Search

In [8]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1783611286, 'localtime': '2026-07-09 08:34'}, 'current': {'last_updated_epoch': 1783610100, 'last_updated': '2026-07-09 08:15', 'temp_c': 13.3, 'temp_f': 55.9, 'is_day': 1, 'condition': {'text': 'Overcast', 'icon': '//cdn.weatherapi.com/weather/64x64/day/122.png', 'code': 1009}, 'wind_mph': 5.4, 'wind_kph': 8.6, 'wind_degree': 191, 'wind_dir': 'S', 'pressure_mb': 1016.0, 'pressure_in': 30.0, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 90, 'cloud': 100, 'feelslike_c': 12.8, 'feelslike_f': 55.0, 'windchill_c': 12.8, 'windchill_f': 55.0, 'heatindex_c': 13.4, 'heatindex_f': 56.1, 'dewpoint_c': 12.8, 'dewpoint_f': 55.0, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 1.1, 'gust_mph': 7.0, 'gust_kph': 11.3, 'will_it_rain': 0, 'chance_of_rain': 33, 'will_it_snow': 0, 'chance_of_snow': 0}}


In [9]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1783611286,
        "localtime": "2026-07-09 08:34"
    },
    "current": {
        "last_updated_epoch": 1783610100,
        "last_updated": "2026-07-09 08:15",
        "temp_c": 13.3,
        "temp_f": 55.9,
        "is_day": 1,
        "condition": {
            "text": "Overcast",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/122.png",
            "code": 1009
        },
        "wind_mph": 5.4,
        "wind_kph": 8.6,
        "wind_degree": 191,
        "wind_dir": "S",
        "pressure_mb": 1016.0,
        "pressure_in": 30.0,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 90,
        "cloud": 100,
        "feelslike_c": 12.8,
        "feelslike_f": 55.0,
        "windchill_c": 12.8,
        "win